# Transformation explorer

Plots a TIMIT WAV file's raw waveform next to its spectrogram, featurized like `src/debug_items.py` does
(`compute_mel_spectrogram` / `compute_linear_spectrogram` from `src/data/features.py`, then `apply_drc`).
The widgets start at the `TransformationConfig` defaults; changing any of them (or the WAV file) re-does the transformation.

The second row shows what a model is fed:

- **Normalized utterance.** The whole utterance after `featurize_waveform`, under the selected `normalization`:
  - `frame`: per-frame z-score. This also removes each frame's level, i.e. the energy contour.
  - `segment`: each band's mean over the drawn segment is subtracted, which removes gain and channel but keeps the energy contour.
  - `none`: no normalization.

  The frames the drawn segment uses are marked.
- **Drawn segment.** One segment of `segment_duration`, drawn with the selected strategy from `DRAW_STRATEGIES` (`src/data/segments.py`): OS, SS, SU, BS{b} (blocks of b frames permuted, order inside each block kept), MIX or RANDBS. It is shown exactly as the model receives it, so `segment` normalization is applied here.
- **Shuffle pattern.** Which original frame each segment position comes from.

In [13]:
import html
import sys
from dataclasses import replace
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
import soundfile as sf
from IPython.display import Audio, display
from plotly.subplots import make_subplots
import plotly.io as pio; pio.renderers.default = "notebook"

ROOT = Path.cwd().parent  # notebook runs from doc/
sys.path.insert(0, str(ROOT))

from src.config import DataConfig, TransformationConfig
from src.data.dataset import featurize_waveform
from src.data.features import apply_drc, compute_linear_spectrogram, compute_mel_spectrogram
from src.data.segments import DRAW_STRATEGIES

In [14]:
timit_root = Path("~/.cache/datasets/timit").expanduser()
# Labelled relative to the TIMIT/ folder, e.g. "TRAIN/DR1/FCJF0/SA1.WAV".
wav_paths = [(str(p.relative_to(p.parents[3])), p) for p in sorted(timit_root.rglob("*")) if p.suffix.lower() == ".wav"]
len(wav_paths)

6300

In [15]:
defaults = TransformationConfig()
wav_path = widgets.Dropdown(options=wav_paths, description="WAV file", layout=widgets.Layout(width="500px"))
frame_length_s = widgets.FloatText(value=defaults.frame_length_s, step=0.001, description="frame_length_s")
frame_step_s = widgets.FloatText(value=defaults.frame_step_s, step=0.001, description="frame_step_s")
nfft = widgets.IntText(value=defaults.nfft, description="nfft")
n_mels = widgets.IntText(value=defaults.n_mels, description="n_mels")
fmin = widgets.BoundedFloatText(value=defaults.fmin, min=0, max=defaults.sample_rate, description="fmin")
fmax = widgets.BoundedFloatText(value=defaults.fmax, min=defaults.fmin, max=defaults.sample_rate, description="fmax")
type_ = widgets.Dropdown(options=[("mel", "mel"), ("lin", "linear")], value=defaults.type, description="type")
window = widgets.Dropdown(options=["hann", "hamming"], value=defaults.window, description="window")
normalization = widgets.Dropdown(options=["frame", "segment", "none"], value=defaults.normalization, description="normalization")
strategy = widgets.Dropdown(options=list(DRAW_STRATEGIES), value="OS", description="strategy")
segment_duration = widgets.BoundedFloatText(value=DataConfig().segment_duration, min=0.1, step=0.1, description="segment [s]")
seed = widgets.IntText(value=0, description="seed")

# Keep fmin <= fmax: each bounds the other.
widgets.dlink((fmin, "value"), (fmax, "min"))
widgets.dlink((fmax, "value"), (fmin, "max"))


# The figure (as an iframe) and the audio player go into HTML widgets: plotly's and Audio's own
# notebook outputs don't render inside an Output widget in PyCharm.
plot = widgets.HTML()
player = widgets.HTML()


def show(wav_path, frame_length_s, frame_step_s, nfft, n_mels, fmin, fmax, type_, window,
         normalization, strategy, segment_duration, seed):
    waveform, sample_rate = sf.read(str(wav_path), dtype="float32")
    assert sample_rate == defaults.sample_rate, f"{wav_path} is {sample_rate} Hz, config expects {defaults.sample_rate} Hz"
    config = replace(defaults, frame_length_s=frame_length_s, frame_step_s=frame_step_s, nfft=nfft,
                     n_mels=n_mels, fmin=fmin, fmax=fmax, type=type_, window=window, normalization=normalization)
    # FFT magnitude of every frame (the linear spectrogram); the mel spectrogram is computed from it.
    stft = apply_drc(compute_linear_spectrogram(waveform, config))
    if type_ == "linear":
        features = stft
    else:
        features = apply_drc(compute_mel_spectrogram(waveform, config))

    # What the model is fed: the utterance as featurize_waveform normalizes it ("frame" only), then one
    # segment drawn with `strategy`. Drawing from an array of frame indices with an identically seeded
    # Generator makes the same random choices, which yields the segment's source frames.
    normalized = featurize_waveform(waveform, config)
    segment_length = DataConfig(segment_duration=segment_duration).segment_length(config)
    if normalized.shape[0] <= segment_length:
        plot.value = f"<b>The utterance ({normalized.shape[0]} frames) must be longer than one segment ({segment_length} frames).</b>"
        player.value = ""
        return
    draw = DRAW_STRATEGIES[strategy]
    source_frames = draw(np.arange(normalized.shape[0])[:, None], segment_length, np.random.default_rng(seed))[:, 0]
    segment = draw(normalized, segment_length, np.random.default_rng(seed))
    if normalization == "segment":
        # As src/models/registry.py's SegmentMeanNormalization does to every model input. The utterance gets the
        # same band means subtracted, so the drawn frames have the same values in both panels.
        band_means = segment.mean(axis=0, keepdims=True)
        segment = segment - band_means
        normalized = normalized - band_means

    # Plotly, so hovering shows the exact values at the mouse position.
    title = f"waveform: {wav_path.relative_to(wav_path.parents[3])}"
    freq_label = "mel bin" if type_ == "mel" else "freq bin"
    frame_times = np.arange(features.shape[0]) * config.frame_step_s
    fig = make_subplots(rows=2, cols=3, horizontal_spacing=0.1, vertical_spacing=0.16, subplot_titles=[
        title, f"FFT per frame: {stft.shape[0]} frames x {stft.shape[1]} bins",
        f"{type_} spectrogram {features.shape[0]} frames x {features.shape[1]} bins",
        f"normalized ({normalization}), frames of the drawn segment marked",
        f"{strategy} segment as the model sees it: {segment.shape[0]} frames",
        "shuffle pattern: source frame per segment position"])

    def colorbar(row, col, title):
        """A colour bar right next to subplot (row, col), as tall as it."""
        subplot = fig.get_subplot(row, col)
        (_, x1), (y0, y1) = subplot.xaxis.domain, subplot.yaxis.domain
        return dict(title_text=title, x=x1 + 0.005, xanchor="left", y=(y0 + y1) / 2, yanchor="middle", len=y1 - y0,
                    thickness=12)

    fig.add_trace(go.Scattergl(
        x=np.arange(len(waveform)) / sample_rate, y=waveform, mode="lines", line_width=0.5, showlegend=False,
        hovertemplate="time: %{x:.4f} s<br>amplitude: %{y:.4f}<extra></extra>",
    ), row=1, col=1)
    fig.add_trace(go.Heatmap(
        z=stft.T, x=frame_times, y=np.arange(stft.shape[1]) * sample_rate / config.nfft,
        colorscale="Magma", colorbar=colorbar(1, 2, "DRC level"),
        hovertemplate="time: %{x:.3f} s<br>frequency: %{y:.1f} Hz<br>DRC level: %{z:.2f}<extra></extra>",
    ), row=1, col=2)
    fig.add_trace(go.Heatmap(
        z=features.T, x=frame_times, y=np.arange(features.shape[1]),
        colorscale="Magma", colorbar=colorbar(1, 3, "DRC level"),
        hovertemplate=f"time: %{{x:.3f}} s<br>{freq_label}: %{{y}}<br>DRC level: %{{z:.2f}}<extra></extra>",
    ), row=1, col=3)
    # Both normalized heatmaps share one colour scale, so equal values get equal colours. Normalized features
    # are centred on 0 (diverging colours); without normalization they're DRC levels.
    level = dict(colorscale="Magma") if normalization == "none" else dict(colorscale="RdBu_r", cmid=0)
    level_name = "DRC level" if normalization == "none" else "normalized level"
    fig.update_layout(coloraxis=dict(**level, colorbar=colorbar(2, 2, level_name)))
    fig.add_trace(go.Heatmap(
        z=normalized.T, x=frame_times, y=np.arange(normalized.shape[1]), coloraxis="coloraxis",
        hovertemplate=f"time: %{{x:.3f}} s<br>{freq_label}: %{{y}}<br>level: %{{z:.2f}}<extra></extra>",
    ), row=2, col=1)
    fig.add_trace(go.Scattergl(
        x=np.unique(source_frames) * config.frame_step_s, y=np.full(len(np.unique(source_frames)), -2),
        mode="markers", marker=dict(color="lime", size=4, symbol="line-ns-open"), showlegend=False,
        hovertemplate="drawn frame at %{x:.3f} s<extra></extra>",
    ), row=2, col=1)
    fig.add_trace(go.Heatmap(
        z=segment.T, x=np.arange(segment.shape[0]), y=np.arange(segment.shape[1]), coloraxis="coloraxis",
        customdata=np.tile(source_frames, (segment.shape[1], 1)),
        hovertemplate=f"position: %{{x}}<br>source frame: %{{customdata}}<br>{freq_label}: %{{y}}<br>level: %{{z:.2f}}<extra></extra>",
    ), row=2, col=2)
    fig.add_trace(go.Scattergl(
        x=np.arange(len(source_frames)), y=source_frames, mode="markers", marker=dict(size=4), showlegend=False,
        hovertemplate="position: %{x}<br>source frame: %{y}<extra></extra>",
    ), row=2, col=3)
    fig.update_xaxes(title_text="segment position [frame]", row=2, col=2)
    fig.update_xaxes(title_text="segment position [frame]", row=2, col=3)
    fig.update_yaxes(title_text=freq_label, row=2, col=1)
    fig.update_yaxes(title_text=freq_label, row=2, col=2)
    fig.update_yaxes(title_text="source frame", row=2, col=3)
    if type_ == "mel":
        # The part of the FFT spectrum the mel filterbank spreads its filters over.
        for name, f in [("fmin", fmin), ("fmax", fmax)]:
            fig.add_hline(y=f, line=dict(color="cyan", width=1, dash="dash"), annotation_text=name,
                          annotation_font_color="cyan", row=1, col=2)
    fig.update_xaxes(title_text="time [s]", row=1)
    fig.update_xaxes(title_text="time [s]", row=2, col=1)
    fig.update_yaxes(title_text="amplitude", row=1, col=1)
    fig.update_yaxes(title_text="frequency [Hz]", row=1, col=2)
    fig.update_yaxes(title_text=freq_label, row=1, col=3)
    fig.update_layout(height=800, margin=dict(l=60, r=80, t=40, b=50))
    page = fig.to_html(include_plotlyjs="cdn", config=dict(responsive=True))
    plot.value = f'<iframe srcdoc="{html.escape(page)}" style="width: 100%; height: 820px; border: 0"></iframe>'
    player.value = Audio(waveform, rate=sample_rate)._repr_html_()


output = widgets.interactive_output(show, dict(
    wav_path=wav_path, frame_length_s=frame_length_s, frame_step_s=frame_step_s, nfft=nfft, n_mels=n_mels,
    fmin=fmin, fmax=fmax, type_=type_, window=window,
    normalization=normalization, strategy=strategy, segment_duration=segment_duration, seed=seed,
))
display(widgets.VBox([
    wav_path,
    widgets.HBox([frame_length_s, frame_step_s, nfft, n_mels]),
    widgets.HBox([fmin, fmax, type_, window]),
    widgets.HBox([normalization, strategy, segment_duration, seed]),
    plot,
    player,
    output,
]))
